# Landing — S&P 500 tracker prices

Source: Yahoo Finance via `yfinance`. Four tickers tracking the S&P 500 in different
wrappers. **SPY is the benchmark** used in every calculation; the other three feed a
secondary "these trackers should overlap" chart.

Same request shape as the trust pull: monthly bars over the full available history, with
`auto_adjust=False, actions=True` so `Close`, `Adj_Close` and `Dividends` all land.

Expected: **3 of 4 symbols return data**, 915 rows.

In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone

import pandas as pd
import yfinance as yf

CATALOG = "`index-vs-trust-pipeline`"
PRICES_TABLE = f"{CATALOG}.landing.index_prices_raw"
LOG_TABLE = f"{CATALOG}.landing.yf_pull_log_raw"
ASSET_CLASS = "index"

# Matches the declared column order of landing.yf_pull_log_raw.
LOG_SCHEMA = (
    "asset_class string, source_ticker string, requested_symbol string, status string, "
    "row_count int, currency string, first_bar string, last_bar string, message string, "
    "pulled_at timestamp"
)

INDEX_TICKERS = ["SPY", "IVV", "VOO", "SPLG"]

In [0]:
frames = []
log_rows = []
pulled_at = datetime.now(timezone.utc)

for ticker in INDEX_TICKERS:
    # US listings carry no exchange suffix, so the symbol is the ticker.
    symbol = ticker

    try:
        yf_ticker = yf.Ticker(symbol)
        hist = yf_ticker.history(
            period="max", interval="1mo", auto_adjust=False, actions=True
        )
    except Exception as exc:
        log_rows.append(
            (ASSET_CLASS, ticker, symbol, "ERROR", 0, None, None, None, str(exc)[:500], pulled_at)
        )
        print(f"{symbol:6} ERROR   {exc}")
        continue

    # Yahoo returns an empty frame for delisted symbols; log the gap and skip.
    if hist.empty:
        log_rows.append(
            (ASSET_CLASS, ticker, symbol, "NODATA", 0, None, None, None, "no rows returned", pulled_at)
        )
        print(f"{symbol:6} NODATA")
        continue

    # yfinance puts the bar date in the row index, not a column.
    hist = hist.reset_index()

    # Delta rejects spaces in column names, so "Adj Close" cannot be stored as it stands.
    hist.columns = [c.replace(" ", "_") for c in hist.columns]

    # Spark cannot store a timezone-aware pandas timestamp, and yfinance omits the
    # timezone on some responses, where stripping an absent one raises.
    if hist["Date"].dt.tz is not None:
        hist["Date"] = hist["Date"].dt.tz_localize(None)

    # The response does not name the symbol it describes.
    hist.insert(0, "source_ticker", ticker)
    hist.insert(1, "symbol", symbol)

    try:
        currency = yf_ticker.fast_info["currency"]
    except Exception:
        currency = None

    first_bar = str(hist["Date"].min().date())
    last_bar = str(hist["Date"].max().date())

    frames.append(hist)
    log_rows.append(
        (ASSET_CLASS, ticker, symbol, "OK", len(hist), currency, first_bar, last_bar, None, pulled_at)
    )
    print(f"{symbol:6} OK     {len(hist):4} bars  {first_bar} to {last_bar}  {currency}")

print(f"\n{len(frames)} of {len(INDEX_TICKERS)} symbols returned data")

In [0]:
# A total failure would otherwise surface as an obscure pandas error on an empty list.
if not frames:
    raise RuntimeError("Yahoo returned no rows for any symbol -- check connectivity")

index_prices = pd.concat(frames, ignore_index=True)

# Capital_Gains is returned for some instruments and not others, so the column set can
# shift between runs; overwriteSchema lets the table follow it.
(
    spark.createDataFrame(index_prices)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PRICES_TABLE)
)

print(f"wrote {len(index_prices)} rows to {PRICES_TABLE}")

In [0]:
# Delete then append this notebook's slice, so the trust pull's rows survive and a
# re-run cannot double-count.
spark.sql(f"DELETE FROM {LOG_TABLE} WHERE asset_class = '{ASSET_CLASS}'")

(
    spark.createDataFrame(log_rows, LOG_SCHEMA)
    .write.format("delta")
    .mode("append")
    .saveAsTable(LOG_TABLE)
)

print(f"logged {len(log_rows)} symbols to {LOG_TABLE}")

## Verification

In [0]:
%sql
SELECT symbol,
       COUNT(*)    AS bars,
       MIN(`Date`) AS first_bar,
       MAX(`Date`) AS last_bar
FROM `index-vs-trust-pipeline`.landing.index_prices_raw
GROUP BY symbol
ORDER BY symbol;

Expect **3 symbols, 915 bars in total**. The counts differ because the funds are
different ages, which is the data, not a bug:

| symbol | first bar | bars |
|---|---|---|
| SPY | 1993-01 | 405 |
| IVV | 2000-05 | 317 |
| VOO | 2010-09 | 193 |

Counts grow by one per symbol each month. The **first dates are exact** and are the real
check: if SPY starts in 2011 rather than 1993, the history has been cut short somewhere.

**SPLG returns nothing.** Yahoo answers `No data found, symbol may be delisted`, so zero
SPLG rows land and the pull log records the gap. SPLG only ever fed the secondary overlap
chart; SPY, the benchmark every calculation uses, is unaffected.

In [0]:
%sql
-- SPY's dividends are what make a like-for-like total return against the trusts possible.
SELECT COUNT(*)                 AS dividend_bars,
       ROUND(SUM(Dividends), 2) AS total_paid_usd
FROM `index-vs-trust-pipeline`.landing.index_prices_raw
WHERE symbol = 'SPY' AND Dividends > 0;

Expect well over 100 dividend bars — SPY has paid quarterly since 1993. Zero here means
`actions=True` was dropped from the request, and total return would be impossible.

In [0]:
%sql
SELECT asset_class, status, COUNT(*) AS symbols
FROM `index-vs-trust-pipeline`.landing.yf_pull_log_raw
GROUP BY asset_class, status
ORDER BY asset_class, status;

With both pulls run, expect **122 rows**: index OK 3 / NODATA 1, trust OK 100 / NODATA 18.
Run this after both notebooks to confirm neither wiped the other's slice.